In [55]:
import matplotlib.pyplot as pyplot
import numpy as np 
import pandas as pd 

In [56]:
df = pd.read_csv("epl_final.csv")

In [57]:
df.head()

,Season,MatchDate,HomeTeam,AwayTeam,FullTimeHomeGoals,FullTimeAwayGoals,FullTimeResult,HalfTimeHomeGoals,HalfTimeAwayGoals,HalfTimeResult,...,HomeShotsOnTarget,AwayShotsOnTarget,HomeCorners,AwayCorners,HomeFouls,AwayFouls,HomeYellowCards,AwayYellowCards,HomeRedCards,AwayRedCards
0,2000/01,2000-08-19,Charlton,Man City,4,0,H,2,0,H,...,14,4,6,6,13,12,1,2,0,0
1,2000/01,2000-08-19,Chelsea,West Ham,4,2,H,1,0,H,...,10,5,7,7,19,14,1,2,0,0
2,2000/01,2000-08-19,Coventry,Middlesbrough,1,3,A,1,1,D,...,3,9,8,4,15,21,5,3,1,0
3,2000/01,2000-08-19,Derby,Southampton,2,2,D,1,2,A,...,4,6,5,8,11,13,1,1,0,0
4,2000/01,2000-08-19,Leeds,Everton,2,0,H,2,0,H,...,8,6,6,4,21,20,1,3,0,0


In [58]:
df.columns

Index(['Season', 'MatchDate', 'HomeTeam', 'AwayTeam', 'FullTimeHomeGoals',
       'FullTimeAwayGoals', 'FullTimeResult', 'HalfTimeHomeGoals',
       'HalfTimeAwayGoals', 'HalfTimeResult', 'HomeShots', 'AwayShots',
       'HomeShotsOnTarget', 'AwayShotsOnTarget', 'HomeCorners', 'AwayCorners',
       'HomeFouls', 'AwayFouls', 'HomeYellowCards', 'AwayYellowCards',
       'HomeRedCards', 'AwayRedCards'],
      dtype='str')

In [59]:
df.HomeTeam.nunique()

46

In [60]:
len(df)

9380

In [61]:
data = {}

for col in df.columns:
    data[col] = {'dtype' : df[col].dtype, 'nunique' : df[col].nunique(), 'null' : df[col].isnull().sum()}

In [62]:
data

{'Season': {'dtype': <StringDtype(storage='python', na_value=nan)>,
  'nunique': 25,
  'null': np.int64(0)},
 'MatchDate': {'dtype': <StringDtype(storage='python', na_value=nan)>,
  'nunique': 2599,
  'null': np.int64(0)},
 'HomeTeam': {'dtype': <StringDtype(storage='python', na_value=nan)>,
  'nunique': 46,
  'null': np.int64(0)},
 'AwayTeam': {'dtype': <StringDtype(storage='python', na_value=nan)>,
  'nunique': 46,
  'null': np.int64(0)},
 'FullTimeHomeGoals': {'dtype': dtype('int64'),
  'nunique': 10,
  'null': np.int64(0)},
 'FullTimeAwayGoals': {'dtype': dtype('int64'),
  'nunique': 10,
  'null': np.int64(0)},
 'FullTimeResult': {'dtype': <StringDtype(storage='python', na_value=nan)>,
  'nunique': 3,
  'null': np.int64(0)},
 'HalfTimeHomeGoals': {'dtype': dtype('int64'),
  'nunique': 6,
  'null': np.int64(0)},
 'HalfTimeAwayGoals': {'dtype': dtype('int64'),
  'nunique': 6,
  'null': np.int64(0)},
 'HalfTimeResult': {'dtype': <StringDtype(storage='python', na_value=nan)>,
  'nuniqu

In [63]:
df = df.sort_values("MatchDate")

In [64]:
df.columns

Index(['Season', 'MatchDate', 'HomeTeam', 'AwayTeam', 'FullTimeHomeGoals',
       'FullTimeAwayGoals', 'FullTimeResult', 'HalfTimeHomeGoals',
       'HalfTimeAwayGoals', 'HalfTimeResult', 'HomeShots', 'AwayShots',
       'HomeShotsOnTarget', 'AwayShotsOnTarget', 'HomeCorners', 'AwayCorners',
       'HomeFouls', 'AwayFouls', 'HomeYellowCards', 'AwayYellowCards',
       'HomeRedCards', 'AwayRedCards'],
      dtype='str')

In [65]:
teams = set(df.HomeTeam.unique())
away_teams = set(df.AwayTeam.unique())

In [66]:
len(teams), len(away_teams)

(46, 46)

In [67]:
teams == away_teams

True

# **ELO Rating**

Since the dataset is already pretty neat and clean, we will just create an ELO score for the team to help us rate how it has performed over the years. 

We will give all teams an initial ELO rating of 1500, and then adjust it and optimize it based on how each team performs. The following formula and process will be used to determine the right elo scores for each team. 


## **Elo Rating System**

Each team has a rating $R$.


### **Expected Score (Before Match)**

$$
E_A = \frac{1}{1 + 10^{\frac{R_B - R_A}{400}}}
$$

Where:

- $E_A$ = expected probability of Team A winning  
- $R_A$ = rating of Team A  
- $R_B$ = rating of Team B  

### **Rating Update (After Match)**

$$
R_A' = R_A + K \cdot (S_A - E_A)
$$

Where:

- $R_A'$ = updated rating for Team A  
- $K$ = learning rate / sensitivity factor (typically 20–40)  
- $S_A$ = actual result for Team A  

### **Match Result Encoding**

$$
S_A =
\begin{cases}
1 & \text{if Team A wins} \\
0.5 & \text{if draw} \\
0 & \text{if Team A loses}
\end{cases}
$$


### **Key Idea**

- If a team performs **better than expected**, its rating increases  
- If it performs **worse than expected**, its rating decreases

In [68]:
elo = {team: 1500 for team in teams}

In [69]:
def expected_score(ra, rb):
    return 1 / (1 + 10 ** ((rb - ra) / 400))

In [70]:
def get_score(result, is_home):
    if result == "H":
        return 1 if is_home else 0
    elif result == "A":
        return 0 if is_home else 1
    else:
        return 0.5

In [71]:
K = 20

home_elo = []
away_elo = []

for i, row in df.iterrows():
    home = row["HomeTeam"]
    away = row["AwayTeam"]

    Ra = elo[home]
    Rb = elo[away]

    home_elo.append(Ra)
    away_elo.append(Rb)

    Eh = expected_score(Ra, Rb)
    Ea = 1 - Eh

    Sh = get_score(row["FullTimeResult"], is_home=True)
    Sa = get_score(row["FullTimeResult"], is_home=False)

    elo[home] = Ra + K * (Sh - Eh)
    elo[away] = Rb + K * (Sa - Ea)

In [72]:
df.columns

Index(['Season', 'MatchDate', 'HomeTeam', 'AwayTeam', 'FullTimeHomeGoals',
       'FullTimeAwayGoals', 'FullTimeResult', 'HalfTimeHomeGoals',
       'HalfTimeAwayGoals', 'HalfTimeResult', 'HomeShots', 'AwayShots',
       'HomeShotsOnTarget', 'AwayShotsOnTarget', 'HomeCorners', 'AwayCorners',
       'HomeFouls', 'AwayFouls', 'HomeYellowCards', 'AwayYellowCards',
       'HomeRedCards', 'AwayRedCards'],
      dtype='str')

In [73]:
df.FullTimeResult.sample(10)

6210    A
4553    D
8612    A
7050    A
3739    H
3151    D
1751    H
6781    H
7060    A
6538    H
Name: FullTimeResult, dtype: str

In [74]:
df["Elo"] = df["HomeTeam"].map(elo)

In [79]:
df['HomeElo'] = home_elo
df['AwayElo'] = away_elo
df['EloDiff'] = df["HomeElo"] - df["AwayElo"]

In [75]:
df.Elo

0       1461.636013
1       1674.496189
2       1427.957841
3       1293.552821
4       1461.853400
           ...     
9377    1527.621265
9378    1674.496189
9375    1601.115637
9376    1600.432487
9379    1597.300961
Name: Elo, Length: 9380, dtype: float64

In [80]:
df.columns

Index(['Season', 'MatchDate', 'HomeTeam', 'AwayTeam', 'FullTimeHomeGoals',
       'FullTimeAwayGoals', 'FullTimeResult', 'HalfTimeHomeGoals',
       'HalfTimeAwayGoals', 'HalfTimeResult', 'HomeShots', 'AwayShots',
       'HomeShotsOnTarget', 'AwayShotsOnTarget', 'HomeCorners', 'AwayCorners',
       'HomeFouls', 'AwayFouls', 'HomeYellowCards', 'AwayYellowCards',
       'HomeRedCards', 'AwayRedCards', 'Elo', 'HomeElo', 'AwayElo', 'EloDiff'],
      dtype='str')

In [81]:
df.Season.unique()

<StringArray>
['2000/01', '2001/02', '2002/03', '2003/04', '2004/05', '2005/06', '2006/07',
 '2007/08', '2008/09', '2009/10', '2010/11', '2011/12', '2012/13', '2013/14',
 '2014/15', '2015/16', '2016/17', '2017/18', '2018/19', '2019/20', '2020/21',
 '2021/22', '2022/23', '2023/24', '2024/25']
Length: 25, dtype: str

# **Time Encoding**
Since we are using a GBM and not Timeseries, we need to give the model an idea of how to map time into its predictions. 

### **Encoding Seasons**
Since the seasons in the football matches are sequential, one follows another, we can use a primitive labeling of each season by just creating an enumerated list of seasons and then add it it a season encoded column that will be fed into the ML model. We will also add a season year column to help the model know which years predictions its making. 

### **Encoding Time in the current Season, like how far along are we since the season started**
Instead of mapping time by using a timestamp, which would be different for each game, we are going to use some sort of progress map that shows us from the season start, how many months we are into the game, I think we can also use days, but month and year can be sufficient for this model. 

In [82]:
season_map = {s: i for i, s in enumerate(sorted(df["Season"].unique()))}

df["SeasonEncoded"] = df["Season"].map(season_map)

In [83]:
season_map

{'2000/01': 0,
 '2001/02': 1,
 '2002/03': 2,
 '2003/04': 3,
 '2004/05': 4,
 '2005/06': 5,
 '2006/07': 6,
 '2007/08': 7,
 '2008/09': 8,
 '2009/10': 9,
 '2010/11': 10,
 '2011/12': 11,
 '2012/13': 12,
 '2013/14': 13,
 '2014/15': 14,
 '2015/16': 15,
 '2016/17': 16,
 '2017/18': 17,
 '2018/19': 18,
 '2019/20': 19,
 '2020/21': 20,
 '2021/22': 21,
 '2022/23': 22,
 '2023/24': 23,
 '2024/25': 24}

In [84]:
base_year = 2000
df["SeasonYear"] = df["SeasonEncoded"] + base_year

In [85]:
df.SeasonEncoded

0        0
1        0
2        0
3        0
4        0
        ..
9377    24
9378    24
9375    24
9376    24
9379    24
Name: SeasonEncoded, Length: 9380, dtype: int64

In [86]:
df.SeasonYear

0       2000
1       2000
2       2000
3       2000
4       2000
        ... 
9377    2024
9378    2024
9375    2024
9376    2024
9379    2024
Name: SeasonYear, Length: 9380, dtype: int64

In [107]:
df["MatchDate"] = pd.to_datetime(df["MatchDate"])
df = df.sort_values("MatchDate")

df["Month"] = df["MatchDate"].dt.month
df["Date"] = df["MatchDate"].dt.day

In [92]:
df.Month.unique()

array([ 8,  9, 10, 11, 12,  1,  2,  3,  4,  5,  6,  7], dtype=int32)

In [98]:
df.Year.unique()

array([2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010,
       2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021,
       2022, 2023, 2024, 2025], dtype=int32)

In [108]:
df.columns

Index(['Season', 'MatchDate', 'HomeTeam', 'AwayTeam', 'FullTimeHomeGoals',
       'FullTimeAwayGoals', 'FullTimeResult', 'HalfTimeHomeGoals',
       'HalfTimeAwayGoals', 'HalfTimeResult', 'HomeShots', 'AwayShots',
       'HomeShotsOnTarget', 'AwayShotsOnTarget', 'HomeCorners', 'AwayCorners',
       'HomeFouls', 'AwayFouls', 'HomeYellowCards', 'AwayYellowCards',
       'HomeRedCards', 'AwayRedCards', 'Elo', 'HomeElo', 'AwayElo', 'EloDiff',
       'SeasonEncoded', 'SeasonYear', 'Month', 'Date'],
      dtype='str')

In [109]:
df.dtypes

Season                          str
MatchDate            datetime64[us]
HomeTeam                        str
AwayTeam                        str
FullTimeHomeGoals             int64
FullTimeAwayGoals             int64
FullTimeResult                  str
HalfTimeHomeGoals             int64
HalfTimeAwayGoals             int64
HalfTimeResult                  str
HomeShots                     int64
AwayShots                     int64
HomeShotsOnTarget             int64
AwayShotsOnTarget             int64
HomeCorners                   int64
AwayCorners                   int64
HomeFouls                     int64
AwayFouls                     int64
HomeYellowCards               int64
AwayYellowCards               int64
HomeRedCards                  int64
AwayRedCards                  int64
Elo                         float64
HomeElo                     float64
AwayElo                     float64
EloDiff                     float64
SeasonEncoded                 int64
SeasonYear                  

In [110]:
df.FullTimeResult.unique()

<StringArray>
['H', 'A', 'D']
Length: 3, dtype: str

In [111]:
mapping = {"H": 0, "D": 1, "A": 2}
df["Target"] = df["FullTimeResult"].map(mapping)

In [112]:
df.columns

Index(['Season', 'MatchDate', 'HomeTeam', 'AwayTeam', 'FullTimeHomeGoals',
       'FullTimeAwayGoals', 'FullTimeResult', 'HalfTimeHomeGoals',
       'HalfTimeAwayGoals', 'HalfTimeResult', 'HomeShots', 'AwayShots',
       'HomeShotsOnTarget', 'AwayShotsOnTarget', 'HomeCorners', 'AwayCorners',
       'HomeFouls', 'AwayFouls', 'HomeYellowCards', 'AwayYellowCards',
       'HomeRedCards', 'AwayRedCards', 'Elo', 'HomeElo', 'AwayElo', 'EloDiff',
       'SeasonEncoded', 'SeasonYear', 'Month', 'Date', 'Target'],
      dtype='str')